<a href="https://colab.research.google.com/github/willow788/Text-generation-using-recurrent-LSTM/blob/main/model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install tensorflow

In [2]:
import tensorflow as tf
import numpy as np
import pandas as pd
import random
import sys
print('all assets imported'
)
print('TensorFlow version:', tf.__version__)

all assets imported
TensorFlow version: 2.20.0


In [6]:
df = pd.read_csv('/content/train.csv')


text = " ".join(df['text'].dropna().astype(str).tolist()).lower()
length = len(text)
print(f'Text length: {length} characters')

Text length: 35695884 characters


In [7]:
vocab = sorted(set(text))
vocab_size = len(vocab)
print(f'Vocabulary size: {vocab_size} characters')

char2indx = { char: i for i, char in enumerate(vocab) }
indx2char = np.array(vocab)
text_as_int = np.array([char2indx[c] for c in text])

Vocabulary size: 104 characters


In [8]:
seq_length = 100

char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)

seq = char_dataset.batch(seq_length + 1, drop_remainder=True)

def split_input_target(chunk):
    return chunk[:-1], chunk[1:]

dataset = seq.map(split_input_target)

batch_size = 64
buffer_size = 10000

dataset = dataset.shuffle(buffer_size).batch(batch_size, drop_remainder=True)



In [9]:
#making the model
vocab_size = len(vocab)

#embedding size
embed_dim = 64
rnn_units = 256

#model
model = tf.keras.Sequential([
    #embedding layer
    tf.keras.layers.Embedding(vocab_size, embed_dim, input_shape=(None,)),

    #lstm layer
    tf.keras.layers.LSTM(rnn_units, return_sequences=True,
                         recurrent_initializer = 'glorot_uniform'),

    #dense layer
    tf.keras.layers.Dense(vocab_size)

])

def loss(labels, logits):
    loss = tf.keras.losses.sparse_categorical_crossentropy(labels, logits, from_logits=True)
    return loss

model.compile(optimizer='adam', loss = loss)

#print the model summary
model.summary()




/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:103: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, None, 64)       │         6,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, None, 256)      │       328,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, None, 104)      │        26,728 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 362,088 (1.38 MB)

 Trainable params: 362,088 (1.38 MB)

 Non-trainable params: 0 (0.00 B)

In [10]:
#training the model using gpu if available

if tf.config.list_physical_devices('GPU'):
    print('GPU is available. Training on GPU.')
else:
    print('GPU is not available. Training on CPU.')


Epochs = 20
history = model.fit(dataset, epochs = Epochs)
print('Model trained for', Epochs, 'epochs')
print('model training complete')

GPU is available. Training on GPU.
Epoch 1/20
5522/5522 ━━━━━━━━━━━━━━━━━━━━ 103s 18ms/step - loss: 1.6429
Epoch 2/20
5522/5522 ━━━━━━━━━━━━━━━━━━━━ 93s 17ms/step - loss: 1.3156
Epoch 3/20
5522/5522 ━━━━━━━━━━━━━━━━━━━━ 98s 17ms/step - loss: 1.2536
Epoch 4/20
5522/5522 ━━━━━━━━━━━━━━━━━━━━ 93s 17ms/step - loss: 1.2228
Epoch 5/20
5522/5522 ━━━━━━━━━━━━━━━━━━━━ 143s 17ms/step - loss: 1.2042
Epoch 6/20
5522/5522 ━━━━━━━━━━━━━━━━━━━━ 93s 17ms/step - loss: 1.1907
Epoch 7/20
5522/5522 ━━━━━━━━━━━━━━━━━━━━ 95s 17ms/step - loss: 1.1807
Epoch 8/20
5522/5522 ━━━━━━━━━━━━━━━━━━━━ 96s 17ms/step - loss: 1.1726
Epoch 9/20
5522/5522 ━━━━━━━━━━━━━━━━━━━━ 95s 17ms/step - loss: 1.1661
Epoch 10/20
5522/5522 ━━━━━━━━━━━━━━━━━━━━ 98s 17ms/step - loss: 1.1613
Epoch 11/20
5522/5522 ━━━━━━━━━━━━━━━━━━━━ 137s 17ms/step - loss: 1.1564
Epoch 12/20
5522/5522 ━━━━━━━━━━━━━━━━━━━━ 93s 17ms/step - loss: 1.1527
Epoch 13/20
5522/5522 ━━━━━━━━━━━━━━━━━━━━ 94s 17ms/step - loss: 1.1492
Epoch 14/20
5522/5522 ━━━━━━━━━━━━━

In [23]:
#generating text
def generating_text(model, start_string, char2indx, indx2char, num_generate = 100, temperature = 1.0):

  input_eval = [char2indx.get(s, 0) for s in start_string.lower()]
  input_eval = tf.expand_dims(input_eval, 0)

  text_generated = []

  model.layers[1].reset_states()

  for _ in range(num_generate):
    predictions = model(input_eval)
    predictions = tf.squeeze(predictions, 0)
    #dividing by temp
    predictions = predictions / temperature

    #now sampling the next char
    predicted_id = tf.random.categorical(predictions, num_samples = 1)[-1, 0].numpy()

    #for n+1
    input_eval= tf.expand_dims([predicted_id], 0)
    text_generated.append(indx2char[predicted_id])

  return start_string + ''.join(text_generated)

print('text generation complete')

# Assuming `model` is already defined and trained, and `idx2char` is accessible
# Calling the function with example parameters
print(generating_text(model, start_string = "ophelia", char2indx=char2indx, indx2char=indx2char, num_generate  = 500, temperature = 0.5))

text generation complete
opheliathe blathe sofon as thin tong tin won win g ulon the. pe t an s sthed  thon th tathere he wos ponthe on allinthed terin t t fien the ie are e the an tore t imes cerit pan athemered on mm an the alllon t t t s as ton lare outhe ast and al t ted the ffof the t tin the t thed s athe thin the she wa r ten that s the at an thin alin te tor thon sthe the s thether ore thace ry ar of thedon whe ale tha t thid con a t a there ant an the ther e are an a tes tin be an ce s s s he anas fathexthorerorere th
